# Small CNN for multi-task image classification

Each 224x224 RGB image has 3 categorical attributes:
- **neck_style**
- **fit_silhouette**
- **season**

The network has a shared small CNN backbone and 3 softmax heads, trained
with the sum of 3 CrossEntropy losses. At the end of this notebook there is
a cell that loads the best checkpoint and displays a sample image with its
true labels and the model's predicted labels.

In [ ]:
## 1. Imports
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

## 1.5 Mount Google Drive

Run this in Colab to mount your Drive. After mounting, your files live under
`/content/drive/MyDrive/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Base folder for this project on Drive
DRIVE_BASE = Path('/content/drive/MyDrive/Colab Notebooks/trabajo final vision de computadora/data')
assert DRIVE_BASE.exists(), f"Folder not found on Drive: {DRIVE_BASE}"
print("Drive base folder:", DRIVE_BASE)
print("Contents:", [p.name for p in DRIVE_BASE.iterdir()][:10])

## 2. Config

Edit these paths and hyperparameters to match your setup.
`relative_path` in the CSV is joined with `DATA_DIR` to locate images.

In [ ]:
# Data locations on Google Drive
DATA_DIR = DRIVE_BASE                       # images folder is DATA_DIR / "images"
CSV_PATH = DRIVE_BASE / "labels.csv"        # <-- change if your CSV has a different name

EPOCHS      = 20
BATCH_SIZE  = 32
LR          = 1e-3
VAL_SPLIT   = 0.2
NUM_WORKERS = 2     # Colab usually prefers 2 when reading from Drive
SEED        = 42

# Save checkpoints/label maps back to Drive so they survive runtime restarts.
CHECKPOINT_PATH = DRIVE_BASE / "model.pt"
LABEL_MAP_PATH  = DRIVE_BASE / "label_maps.json"

TARGET_COLS = ["neck_style", "fit_silhouette", "season"]

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Dataset

Reads your CSV (columns: `id, image_url, neck_style, fit_silhouette,
season, relative_path`) and turns each row into a tensor image plus a
dict of integer class indices.

In [ ]:
class DressAttributesDataset(Dataset):
    def __init__(self, df, data_dir, label_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / row["relative_path"]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        targets = {
            col: torch.tensor(self.label_to_idx[col][row[col]], dtype=torch.long)
            for col in TARGET_COLS
        }
        return image, targets


def build_label_maps(df):
    """{col: {str_label: int_index}} with stable, sorted ordering."""
    return {
        col: {c: i for i, c in enumerate(sorted(df[col].dropna().unique().tolist()))}
        for col in TARGET_COLS
    }


## 4. Load CSV and build label encodings

In [ ]:
df = pd.read_csv(CSV_PATH)
required = ["relative_path"] + TARGET_COLS
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")
df = df.dropna(subset=required).reset_index(drop=True)

label_to_idx = build_label_maps(df)
num_classes = {col: len(label_to_idx[col]) for col in TARGET_COLS}

print(f"Total samples: {len(df)}")
for col in TARGET_COLS:
    print(f"  {col}: {num_classes[col]} classes -> {list(label_to_idx[col].keys())}")

# Save mapping so predictions can be decoded later.
with open(LABEL_MAP_PATH, "w") as f:
    json.dump(label_to_idx, f, indent=2)
print(f"Saved label maps to {LABEL_MAP_PATH}")


## 5. Transforms and data loaders

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

n = len(df)
n_val = int(n * VAL_SPLIT)
n_train = n - n_val

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(n, generator=g).tolist()
train_df = df.iloc[perm[:n_train]].reset_index(drop=True)
val_df   = df.iloc[perm[n_train:]].reset_index(drop=True)

train_set = DressAttributesDataset(train_df, DATA_DIR, label_to_idx, train_tf)
val_set   = DressAttributesDataset(val_df,   DATA_DIR, label_to_idx, val_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train samples: {n_train} | Val samples: {n_val}")


## 6. Model

Small CNN backbone (~0.5M params) plus one linear head per attribute.

In [ ]:
class SmallMultiHeadCNN(nn.Module):
    def __init__(self, num_classes_per_head):
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        # Input: 3 x 224 x 224
        self.backbone = nn.Sequential(
            conv_block(3, 32),     # -> 32 x 112 x 112
            conv_block(32, 64),    # -> 64 x 56 x 56
            conv_block(64, 128),   # -> 128 x 28 x 28
            conv_block(128, 128),  # -> 128 x 14 x 14
            conv_block(128, 256),  # -> 256 x 7 x 7
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.dropout = nn.Dropout(0.3)
        self.heads = nn.ModuleDict({
            col: nn.Linear(256, n) for col, n in num_classes_per_head.items()
        })

    def forward(self, x):
        feats = self.dropout(self.backbone(x))
        return {col: head(feats) for col, head in self.heads.items()}


model = SmallMultiHeadCNN(num_classes).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")


## 7. Optimizer, scheduler, loss

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 8. Training / evaluation loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, train):
    model.train() if train else model.eval()
    total_loss, total_samples = 0.0, 0
    correct = {col: 0 for col in TARGET_COLS}

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, targets in loader:
            images  = images.to(device)
            targets = {col: t.to(device) for col, t in targets.items()}

            logits = model(images)
            loss = sum(criterion(logits[col], targets[col]) for col in TARGET_COLS)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            bs = images.size(0)
            total_loss += loss.item() * bs
            total_samples += bs
            for col in TARGET_COLS:
                preds = logits[col].argmax(dim=1)
                correct[col] += (preds == targets[col]).sum().item()

    avg_loss = total_loss / total_samples
    accs = {col: correct[col] / total_samples for col in TARGET_COLS}
    return avg_loss, accs


## 9. Train

In [ ]:
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
    va_loss, va_acc = run_epoch(model, val_loader,   criterion, optimizer, device, train=False)
    scheduler.step()

    tr_str = " ".join(f"{c}={tr_acc[c]:.3f}" for c in TARGET_COLS)
    va_str = " ".join(f"{c}={va_acc[c]:.3f}" for c in TARGET_COLS)
    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"train_loss={tr_loss:.4f} [{tr_str}] | "
          f"val_loss={va_loss:.4f} [{va_str}]")

    if va_loss < best_val:
        best_val = va_loss
        torch.save({
            "model_state": model.state_dict(),
            "num_classes": num_classes,
            "label_to_idx": label_to_idx,
            "epoch": epoch,
            "val_loss": va_loss,
        }, CHECKPOINT_PATH)
        print(f"  -> saved best model to {CHECKPOINT_PATH}")

print(f"Done. Best val_loss={best_val:.4f}")


## 10. Test on a sample: show image, true labels, and predicted labels

Loads the best checkpoint, picks a random image from the validation set,
runs the model, and displays the image alongside true vs predicted
labels for each attribute. Re-run this cell to try another sample.

In [ ]:
# Load the best checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()

# Reverse mapping: int -> str per attribute
idx_to_label = {
    col: {i: c for c, i in m.items()} for col, m in label_to_idx.items()
}

# Pick a random validation sample
i = random.randrange(len(val_df))
row = val_df.iloc[i]
img_path = DATA_DIR / row["relative_path"]

# Original image for display (no normalization)
orig_image = Image.open(img_path).convert("RGB")

# Tensor for the model (same transform as val_tf)
x = val_tf(orig_image).unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(x)
    probs  = {col: torch.softmax(logits[col], dim=1)[0].cpu() for col in TARGET_COLS}
    preds  = {col: int(probs[col].argmax().item()) for col in TARGET_COLS}

true_labels = {col: row[col] for col in TARGET_COLS}
pred_labels = {col: idx_to_label[col][preds[col]] for col in TARGET_COLS}
pred_conf   = {col: float(probs[col][preds[col]]) for col in TARGET_COLS}

# Plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(orig_image)
ax.axis("off")

lines = [f"id: {row['id']}"]
for col in TARGET_COLS:
    ok = "OK" if pred_labels[col] == true_labels[col] else "X "
    lines.append(f"[{ok}] {col}: true={true_labels[col]} | "
                 f"pred={pred_labels[col]} ({pred_conf[col]*100:.1f}%)")
ax.set_title("\n".join(lines), loc="left", fontsize=10)
plt.tight_layout()
plt.show()
